In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: Script en Python para generar mapas de mortalida para la ZMVM. 
# Periodo: 2000-2019
# ==========================================

In [3]:
#UNA IMAGEN TODO EL PERIDO

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import os
import math

# === RUTAS DE ARCHIVOS ===
ruta_defunciones = {
    'respiratorias': "RUTA DEL ARCHIVO",
    'cardiovasculares': "RUTA DEL ARCHIVO",
    'metabolicas': "RUTA DEL ARCHIVO"
}

archivo_poblacion = "RUTA DEL ARCHIVO"
archivo_municipios = "RUTA DEL ARCHIVO"
archivo_shapefile = "RUTA DEL ARCHIVO"

ruta_salida = "RUTA DE SALIDA"
os.makedirs(ruta_salida, exist_ok=True)

nombres_legibles = {
    'respiratorias': 'Respiratorias',
    'cardiovasculares': 'Cardiovasculares',
    'metabolicas': 'Metabólicas'
}

# === CARGA DE ARCHIVOS BASE ===
df_pob = pd.read_csv(archivo_poblacion, encoding='latin1')
df_claves = pd.read_csv(archivo_municipios, encoding='latin1')
gdf_map = gpd.read_file(archivo_shapefile)

# =========================================================
# FUNCIÓN PARA PREPARAR LOS DATOS DE CADA ENFERMEDAD
# =========================================================
def preparar_datos_enfermedad(ruta_csv):
    df = pd.read_csv(ruta_csv, encoding='latin1')
    df = df[df['Anio'].between(2000, 2019)]

    poblacion = df_pob.rename(columns={'CLAVE': 'CVEGEO'})
    poblacion = poblacion[['CVEGEO', 'Anio', 'POB_TOTAL']]

    defunciones = df.groupby(['CVEGEO', 'Anio']).size().reset_index(name='Defunciones')

    df_merge = defunciones.merge(
        poblacion,
        on=['CVEGEO', 'Anio'],
        how='left'
    )

    df_merge['mortalidad'] = (df_merge['Defunciones'] / df_merge['POB_TOTAL']) * 10000

    df_merge = df_merge.merge(
        df_claves[['CVEGEO', 'nom_ent', 'nom_mun']],
        on='CVEGEO',
        how='left'
    )

    gdf_final = gdf_map.merge(df_merge, on='CVEGEO')
    gdf_final = gdf_final[gdf_final['Anio'].between(2000, 2019)]

    return gdf_final

# =========================================================
# CALCULAR ESCALA GLOBAL PARA LAS TRES ENFERMEDADES
# =========================================================
todos_los_gdf = {}
todos_los_valores = []

for enfermedad, ruta in ruta_defunciones.items():
    gdf_temp = preparar_datos_enfermedad(ruta)
    todos_los_gdf[enfermedad] = gdf_temp
    todos_los_valores.extend(gdf_temp['mortalidad'].dropna().tolist())

vmin_global = min(todos_los_valores)
vmax_global = max(todos_los_valores)

# Redondeo para que la barra quede más limpia
vmin_global = math.floor(vmin_global)
vmax_global = math.ceil(vmax_global)

norm_global = mcolors.Normalize(vmin=vmin_global, vmax=vmax_global)
cmap_global = cm.Reds

print(f"Escala global comparable: {vmin_global} a {vmax_global}")

# =========================================================
# FUNCIÓN PARA GRAFICAR TODOS LOS AÑOS EN UNA SOLA FIGURA
# =========================================================
def graficar_mapa_rango(nombre, g, años, cmap, norm, sufijo):
    g = g[g['Anio'].isin(años)].sort_values("Anio")

    cols = 5
    rows = math.ceil(len(años) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(16, 3.8 * rows))
    axes = axes.flatten()

    for i, year in enumerate(años):
        ax = axes[i]
        g_year = g[g['Anio'] == year]

        ax.set_facecolor("#f0f0f0")

        ax.add_patch(plt.Rectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            facecolor='none',
            edgecolor='black',
            linewidth=0.7
        ))

        g_year.plot(
            column='mortalidad',
            cmap=cmap,
            norm=norm,
            linewidth=0.5,
            edgecolor='black',
            ax=ax,
            legend=False
        )

        ax.text(
            0.5, 1.02, f"{year}",
            transform=ax.transAxes,
            fontsize=14,
            fontweight='bold',
            ha='center',
            va='bottom',
            bbox=dict(
                facecolor='white',
                edgecolor='black',
                boxstyle='round,pad=0.45',
                linewidth=0.3
            )
        )

        ax.axis('off')

    # Apagar ejes sobrantes si existieran
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    # Barra de color global
    cax = fig.add_axes([0.92, 0.16, 0.015, 0.6])
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = []
    cbar = fig.colorbar(sm, cax=cax)

    cbar.set_label(
        "Mortalidad (por cada diez mil habitantes)",
        fontsize=11,
        fontweight='bold',
        labelpad=12
    )

    # Título y subtítulo
    fig.suptitle(
        f"MORTALIDAD POR ENFERMEDADES {nombre.upper()}",
        fontsize=17,
        fontweight='bold',
        y=0.98
    )

    fig.text(
        0.5, 0.945,
        "Zona Metropolitana del Valle de México",
        fontsize=15,
        ha='center',
        va='top'
    )

    plt.tight_layout(rect=[0, 0, 0.9, 0.92])

    salida = os.path.join(
        ruta_salida,
        f"mapa_mortalidad_{nombre.lower()}_{sufijo}.png"
    )

    plt.savefig(salida, dpi=300)
    plt.close()

    print(f"✅ Imagen generada: {salida}")

# =========================================================
# GRAFICAR CADA ENFERMEDAD EN UNA SOLA FIGURA 2000-2019
# =========================================================
for enfermedad, gdf_final in todos_los_gdf.items():
    nombre_legible = nombres_legibles.get(enfermedad, enfermedad)

    graficar_mapa_rango(
        nombre_legible,
        gdf_final,
        list(range(2000, 2020)),
        cmap_global,
        norm_global,
        '2000_2019'
    )

Escala global comparable: 0 a 48


/var/folders/8f/xcm8kn2d7csg_55l0rmqbcjc0000gn/T/ipykernel_7999/2253812812.py:176: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.92])


✅ Imagen generada: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MORTALIDAD/MAPAS/mapa_mortalidad_respiratorias_2000_2019.png


/var/folders/8f/xcm8kn2d7csg_55l0rmqbcjc0000gn/T/ipykernel_7999/2253812812.py:176: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.92])


✅ Imagen generada: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MORTALIDAD/MAPAS/mapa_mortalidad_cardiovasculares_2000_2019.png


/var/folders/8f/xcm8kn2d7csg_55l0rmqbcjc0000gn/T/ipykernel_7999/2253812812.py:176: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.92])


✅ Imagen generada: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/MORTALIDAD/MAPAS/mapa_mortalidad_metabólicas_2000_2019.png
